<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_7/Day_5%20/Daily_challenge/Daily_challenge_w7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Daily Challenge : Approach to Complex SQLquery Building in Kaggle
===

**1. Load and Explore the Data**

In [ ]:
import zipfile
import os
import sqlite3
import pandas as pd

zip_path = 'Approach to Complex SQLquery Building in Kaggle.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('ipl_data')

conn = sqlite3.connect('ipl_data/database.sqlite')

In [ ]:
master_table = pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table';", conn)
print(master_table[['name', 'tbl_name']])

In [ ]:
tables = master_table['name'].tolist()

for table in tables:
    cols_info = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print(f"Table: {table}")
    print(f"Columns: {cols_info['name'].tolist()}\n")

**2. Select All Columns from Player’s Table:**

In [ ]:
query_2 = "SELECT * FROM Player_Match"

df_player_match = pd.read_sql_query(query_2, conn)

df_player_match.head()

**3. Batsman vs Runs:**

In [ ]:
query_3 = """
SELECT
    p.Player_Name,
    SUM(bs.Runs_Scored) AS Total_Runs
FROM
    Player p
JOIN
    Ball_by_Ball bb ON p.Player_Id = bb.Striker
JOIN
    Batsman_Scored bs ON bb.Match_Id = bs.Match_Id
    AND bb.Over_Id = bs.Over_Id
    AND bb.Ball_Id = bs.Ball_Id
    AND bb.Innings_No = bs.Innings_No
GROUP BY
    p.Player_Id, p.Player_Name
ORDER BY
    Total_Runs DESC;
"""

df_batsman_runs = pd.read_sql_query(query_3, conn)
df_batsman_runs.head(10)

**4. Fifties and Hundreds:**

In [ ]:
query_4 = """
SELECT
    p.Player_Name,
    SUM(CASE WHEN Match_Runs >= 100 THEN 1 ELSE 0 END) AS Hundreds,
    SUM(CASE WHEN Match_Runs >= 50 AND Match_Runs < 100 THEN 1 ELSE 0 END) AS Fifties
FROM (
    -- Subquery: Calculate total runs per player per match
    SELECT
        bb.Striker AS Player_Id,
        bb.Match_Id,
        SUM(bs.Runs_Scored) AS Match_Runs
    FROM
        Ball_by_Ball bb
    JOIN
        Batsman_Scored bs ON bb.Match_Id = bs.Match_Id
        AND bb.Over_Id = bs.Over_Id
        AND bb.Ball_Id = bs.Ball_Id
        AND bb.Innings_No = bs.Innings_No
    GROUP BY
        bb.Striker, bb.Match_Id
) AS Player_Match_Runs
JOIN
    Player p ON Player_Match_Runs.Player_Id = p.Player_Id
GROUP BY
    p.Player_Id, p.Player_Name
ORDER BY
    Hundreds DESC, Fifties DESC;
"""

df_milestones = pd.read_sql_query(query_4, conn)
df_milestones.head(10)

**5. Best Bowling Figures:**

In [ ]:
query_5 = """
SELECT
    p.Player_Name,
    MAX(Match_Wickets) AS Best_Wickets
FROM (
    -- Subquery: Count bowler-credited wickets per match per bowler
    SELECT
        bb.Bowler AS Player_Id,
        bb.Match_Id,
        COUNT(wt.Kind_Out) AS Match_Wickets
    FROM
        Ball_by_Ball bb
    JOIN
        Wicket_Taken wt ON bb.Match_Id = wt.Match_Id
        AND bb.Over_Id = wt.Over_Id
        AND bb.Ball_Id = wt.Ball_Id
        AND bb.Innings_No = wt.Innings_No
    WHERE
        -- Exclude non-bowler wickets
        wt.Kind_Out NOT IN (SELECT Out_Id FROM Out_Type WHERE Out_Name IN ('run out', 'retired hurt', 'obstructing the field'))
    GROUP BY
        bb.Bowler, bb.Match_Id
) AS Bowler_Stats
JOIN
    Player p ON Bowler_Stats.Player_Id = p.Player_Id
GROUP BY
    p.Player_Id, p.Player_Name
ORDER BY
    Best_Wickets DESC;
"""

df_best_bowling = pd.read_sql_query(query_5, conn)
df_best_bowling.head(10)

**6. Comprehensive Career Metrics:** *italicized text*

In [ ]:
query_6 = """
WITH MatchRuns AS (
    -- Subquery for runs per match
    SELECT bb.Striker AS Player_Id, bb.Match_Id, SUM(bs.Runs_Scored) AS Runs
    FROM Ball_by_Ball bb
    JOIN Batsman_Scored bs ON bb.Match_Id = bs.Match_Id
        AND bb.Over_Id = bs.Over_Id
        AND bb.Ball_Id = bs.Ball_Id
        AND bb.Innings_No = bs.Innings_No
    GROUP BY bb.Striker, bb.Match_Id
),
BattingStats AS (
    -- Aggregating batting milestones
    SELECT Player_Id, SUM(Runs) AS Total_Runs,
           SUM(CASE WHEN Runs >= 100 THEN 1 ELSE 0 END) AS Hundreds,
           SUM(CASE WHEN Runs >= 50 AND Runs < 100 THEN 1 ELSE 0 END) AS Fifties
    FROM MatchRuns
    GROUP BY Player_Id
),
MatchWickets AS (
    -- Subquery for wickets per match
    SELECT bb.Bowler AS Player_Id, bb.Match_Id, COUNT(wt.Kind_Out) AS Wickets
    FROM Ball_by_Ball bb
    JOIN Wicket_Taken wt ON bb.Match_Id = wt.Match_Id
        AND bb.Over_Id = wt.Over_Id
        AND bb.Ball_Id = wt.Ball_Id
        AND bb.Innings_No = wt.Innings_No
    WHERE wt.Kind_Out NOT IN (SELECT Out_Id FROM Out_Type WHERE Out_Name IN ('run out', 'retired hurt', 'obstructing the field'))
    GROUP BY bb.Bowler, bb.Match_Id
),
BowlingStats AS (
    -- Finding career-best wickets
    SELECT Player_Id, MAX(Wickets) AS Best_Bowling
    FROM MatchWickets
    GROUP BY Player_Id
)
-- Final Combine
SELECT
    p.Player_Name,
    COALESCE(b.Total_Runs, 0) AS Total_Runs,
    COALESCE(b.Hundreds, 0) AS Hundreds,
    COALESCE(b.Fifties, 0) AS Fifties,
    COALESCE(bo.Best_Bowling, 0) AS Best_Bowling_Wickets
FROM Player p
LEFT JOIN BattingStats b ON p.Player_Id = b.Player_Id
LEFT JOIN BowlingStats bo ON p.Player_Id = bo.Player_Id
ORDER BY Total_Runs DESC;
"""

df_comprehensive = pd.read_sql_query(query_6, conn)
df_comprehensive.head(10)